# Tensor-seeded radial +1 Gaussian-pairing optimizer

Maximize the squared Gaussian pairing divided by the product of L1 norms, reporting its dth root. The reference Gaussian is L2-normalized. The searched functions are radial Fourier +1 eigenfunctions; no origin condition is imposed.

For each requested (D,N), this notebook first optimizes lower-dimensional candidates, tensors and radializes them, and compares different dimension splits. Near-equal splits use floor/ceiling dimensions, including odd D. The full tensor seed fits the requested mode budget: 1 + sum(N_i-1) <= N. Seeds are not truncated.

Each seed has an inherited certified norm bound from its factors. Optionally its actual radial norm is certified more tightly. The code then fits corrections around the exact seed, using high-precision seed evaluations. The inherited seed bound survives any failed or worse refinement. A direct target-dimensional solve is optional, and runs only after the tensor comparisons.

**Only `reported_root_lower` is the result to cite.** It is rounded downward and is a certified lower bound for the supremum, not a claim of global optimality. Candidate trees retain exact tensor recipes and binary-float corrections; never replace an exact tensor tree by naively rounded target coefficients and assume the same certificate applies.

`TENSOR_FACTORS=[2,3,4]` compares near-equal 2-,3-,4-block splits. `TENSOR_DEPTH=1` directly optimizes each lower-dimensional block. Increase depth for recursive tensor seeds inside those blocks. `EXTRA_SPLITS` allows custom dimension decompositions. Repeated subproblems are cached during the run; all candidate recipes and certificates are also saved.

This can be substantially more expensive than a direct solve. For one final mode count set `MODE_COUNTS=[N_MAX]`. `CERTIFY_TENSOR_NORM=False` skips optional tightening of pure tensor norms, but retains the certified inherited bound. Corrections still require certification. `CHILD_MODE_CAP` can limit the lower-dimensional search sizes. Numerical or precision failures are reported and cannot replace better certified candidates.


In [1]:
%pip install -q numpy scipy python-flint


In [2]:
from pathlib import Path
import sys
D = 220
N_MAX = 256
MODE_COUNTS = [N_MAX]  # Set None for 8,16,32,...,N_MAX, or supply a list.
TENSOR_FACTORS = [2, 3, 4]
TENSOR_DEPTH = 1
EXTRA_SPLITS = {}  # Example: {250: [[100,100,50]]}
CHILD_MODE_CAP = None  # Example: 64; None uses each allocated mode budget.
SEED_CORRECTION_MODES = 32  # Set N_MAX to allow all correction directions.
COMPARE_DIRECT = False    # Optional direct target solve AFTER the tensor searches.
CERTIFY_TENSOR_NORM = True # Tighten inherited bounds using the actual radial norm.
QUAD_POINTS = 8000        # Lower-dimensional/direct fits
SEED_QUAD_POINTS = 1200   # High-precision tensor-seed affine refinement
CERT_DIGITS = 1200
CERT_ROOT_GAP = 5e-6
SOLVER_TIME_LIMIT = 120
AUTO_DOWNLOAD_CSV = True
AUTO_DOWNLOAD_BUNDLE = False
BASE = Path('/content') if Path('/content').exists() else Path.cwd()
MODULE_DIR = BASE / 'tensor_eigenfunction_modules'
MODULE_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(MODULE_DIR))
OUTPUT_DIR = BASE / f'tensor_eigenfunction_results_d{D}'


In [3]:
# Embedded source: certificate.py
source = r'''
"""Arb enclosure of the scaled radial L1 integral. No sampled maximum assumption."""
import math,time,heapq
from decimal import Decimal,ROUND_FLOOR
from flint import arb,ctx

def up(v):return math.nextafter(float(v.upper()),math.inf)
def down(v):return math.nextafter(float(v.lower()),-math.inf)
def addjet(a,b):return tuple(x+y for x,y in zip(a,b))
def muljet(a,b):
 return (a[0]*b[0],a[1]*b[0]+a[0]*b[1],a[2]*b[0]+2*a[1]*b[1]+a[0]*b[2])

def jet(x,d,c):
 a=arb(d)/2;k=d-1
 power=(x**k,arb(k)*x**(k-1) if k else arb(0),arb(k*(k-1))*x**(k-2) if k>=2 else arb(0))
 e=(-x*x).exp();expo=(e,-2*x*e,(4*x*x-2)*e)
 u=tuple(v*2/a.gamma() for v in muljet(power,expo));out=tuple(c[0]*v for v in u)
 if len(c)==1:return out
 q=(a-2*x*x,-4*x,arb(-4));v=tuple(t/a.sqrt() for t in muljet(q,u))
 for n in range(1,2*len(c)-2):
  den=((n+1)*(n+a)).sqrt();s=(n*(n+a-1)).sqrt()/den
  q=((2*n+a-2*x*x)/den,-4*x/den,-4/den)
  new=tuple(t-s*old for t,old in zip(muljet(q,v),u));u,v=v,new
  if (n+1)%2==0:out=tuple(t+c[(n+1)//2]*val for t,val in zip(out,v))
 return out

def tail_bound(d,c,R):
 a=arb(d)/2;N=len(c);coeff=[arb(0) for _ in range(2*N-1)]
 norm=arb(1);binom=arb(1)
 for n in range(2*N-1):
  if n:
   norm*= (arb(n)/(n+a-1)).sqrt();binom*= (n+a-1)/n
  if n%2:continue
  t=binom*norm*c[n//2]
  for j in range(n+1):
   coeff[j]+=t
   if j<n:t*= -2*arb(n-j)/((a+j)*(j+1))
 z=arb(R)**2;total=arb(0);global_bound=arb(0);moment=arb(1)
 for j,v in enumerate(coeff):
  if j:moment*=a+j-1
  global_bound+=abs(v)*moment
  s=a+j
  if z.lower()<=max(arb(0),(s-1)).upper():raise ArithmeticError('Tail radius too small.')
  denom=1-max(arb(0),s-1)/z
  total+=abs(v)*(z**(s-1))*(-z).exp()/a.gamma()/denom
 return total,global_bound

def certify(d,coefficients,max_seconds=60,max_leaves=30000,root_gap=5e-6,digits=160,progress_seconds=10):
 ctx.dps=digits;c=[arb(float(v)) for v in coefficients]
 # Every stored float coefficient is taken as the exact binary float defining the candidate.
 if c[0]!=1:raise ValueError('Certification requires coefficient c0 exactly1.')
 start=time.time();R=math.sqrt(d/2+4*len(c))+10
 tail,global_bound=tail_bound(d,c,R)
 counter=0
 def leaf(left,right):
  nonlocal counter
  counter+=1;mid=(left+right)/2;half=(right-left)/2
  # Exact binary endpoints, center and radius: enclose the complete interval.
  l=arb(left);r=arb(right);m=(l+r)/2;h=(r-l)/2;width=2*h
  interval=arb(m,h.upper())
  f,fp,_=jet(m,d,c);M=abs(jet(interval,d,c)[2])
  if abs(f).lower()>=(abs(fp)*h).upper():linear=width*abs(f)
  elif abs(fp).lower()>0 and abs(f).upper()<(abs(fp)*h).lower():linear=(f*f+(fp*h)**2)/abs(fp)
  else:
   # General triangle bound; the lower bound is deliberately0 when signs are uncertain.
   linear=None
  remainder=M*width**3/24
  if linear is None:
   lo=0.;hi=up(width*abs(f)+h*h*abs(fp)+remainder)
  else:
   lo=max(0.,down(linear-remainder));hi=up(linear+remainder)
  hi=min(hi,up(global_bound));lo=min(lo,hi)
  return (-(hi-lo),counter,left,right,lo,hi)
 heap=[leaf(0.,R)];last=start;reason='budget'
 while True:
  # Arb summation prevents floating accumulation from weakening either direction.
  lower=sum((arb(item[4]) for item in heap),arb(0))
  upper=sum((arb(item[5]) for item in heap),arb(0))+tail
  upper_value=arb(up(upper))
  root_lower=( -arb(2).log()/2-arb(2)/d*upper_value.log()).exp()
  low=down(root_lower)
  if lower.lower()>0:
   root_upper=(-arb(2).log()/2-arb(2)/d*arb(down(lower)).log()).exp()
   high=up(root_upper)
  else:high=math.inf
  elapsed=time.time()-start
  if high-low<=root_gap:reason='root_gap';break
  if elapsed>=max_seconds or len(heap)>=max_leaves:break
  if elapsed-last>=progress_seconds:
   print(f'CERT d={d}, N={len(c)}: candidate root in [{low:.9f}, {high:.9f}]; leaves={len(heap)}',flush=True);last=time.time()
  # Split batches so rigorous full summations do not dominate the runtime.
  for _ in range(min(32,max_leaves-len(heap))):
   _,_,left,right,lo,hi=heapq.heappop(heap);mid=(left+right)/2
   if mid==left or mid==right:
    heapq.heappush(heap,(0.,counter,left,right,lo,hi));reason='resolution';break
   heapq.heappush(heap,leaf(left,mid));heapq.heappush(heap,leaf(mid,right))
  if reason=='resolution':break
 # Save exactly the upper bound used to compute the lower root, as a hexadecimal float.
 return {'root_lower':low,'candidate_root_upper':high if math.isfinite(high) else None,
         'scaled_l1_upper_hex':float(upper_value).hex(),'leaves':len(heap),'seconds':time.time()-start,
         'stop_reason':reason,'digits':digits,'tail_upper':up(tail),'radius':R}

def rounded_lower(value,places=8):
 return str(Decimal.from_float(value).quantize(Decimal(1).scaleb(-places),rounding=ROUND_FLOOR))

'''
(MODULE_DIR / 'certificate.py').write_text(source)


4667

In [4]:
# Embedded source: root_certificate.py
source = r'''
"""Rigorous L1 enclosure by Arb complex-root isolation and incomplete gamma integrals."""
import time,math
from flint import arb,acb_poly,ctx
from certificate import up,down

def polynomial(d,c):
 a=arb(d)/2;C=[arb(0) for _ in range(2*len(c)-1)];norm=arb(1);binom=arb(1)
 for n in range(len(C)):
  if n:norm*=(arb(n)/(n+a-1)).sqrt();binom*=(n+a-1)/n
  if n%2:continue
  t=binom*norm*arb(c[n//2])
  for j in range(n+1):
   C[j]+=t
   if j<n:t*= -2*arb(n-j)/((a+j)*(j+1))
 while len(C)>1 and C[-1]==0:C.pop()
 return C

def primitive_terms(z,a,n):
 if z==0:return [arb(0)]*n
 I=z.gamma_lower(a,regularized=1);q=(a*z.log()-z-a.lgamma()).exp();out=[I]
 for j in range(n-1):
  I=(a+j)*I-q;q*=z;out.append(I)
 return out

def certify_roots(d,c,digits=400,root_gap=5e-6):
 start=time.time();ctx.dps=digits;a=arb(d)/2
 if arb(c[0])!=1:raise ValueError('Requires c0=1 exactly.')
 C=polynomial(d,c)
 if len(C)==1:roots=[]
 else:roots=acb_poly(C).roots(tol=arb(10)**(-digits//2),maxprec=int(digits*5))
 if len(roots)!=len(C)-1:raise ArithmeticError('Not all polynomial roots were isolated.')
 windows=[]
 for r in roots:
  # Any positive real root must belong to one of these real projections.
  if r.imag.contains(0) and r.real.upper()>0:
   lo=max(arb(0),r.real.lower());hi=r.real.upper();windows.append([lo,hi])
 windows.sort(key=lambda v:v[0]);merged=[]
 for lo,hi in windows:
  if merged and lo<=merged[-1][1]:
   merged[-1][0]=min(lo,merged[-1][0]);merged[-1][1]=max(hi,merged[-1][1])
  else:merged.append([lo,hi])
 moment=arb(1);total=C[0]
 for j in range(1,len(C)):
  moment*=a+j-1;total+=C[j]*moment
 # On intervals outside all root windows the real polynomial has constant sign.
 def integ(terms):return sum((v*t for v,t in zip(C,terms)),arb(0))
 outside=arb(0);uncertain=arb(0);previous_integral=arb(0)
 for lo,hi in merged:
  lower_terms=primitive_terms(lo,a,len(C));upper_terms=primitive_terms(hi,a,len(C))
  left=integ(lower_terms);right=integ(upper_terms)
  outside+=abs(left-previous_integral)
  uncertain+=sum((abs(v)*abs(y-x) for v,x,y in zip(C,lower_terms,upper_terms)),arb(0))
  previous_integral=right
 outside+=abs(total-previous_integral)
 upper=arb(up(outside+uncertain));lower=max(arb(0),outside.lower())
 if not upper.is_finite() or upper<=0:raise ArithmeticError('Nonfinite L1 enclosure; increase CERT_DIGITS.')
 root_lower=down((-arb(2).log()/2-arb(2)/d*upper.log()).exp())
 root_upper=up((-arb(2).log()/2-arb(2)/d*lower.log()).exp()) if lower>0 else None
 return {'root_lower':root_lower,'candidate_root_upper':root_upper,
         'scaled_l1_upper_hex':float(upper).hex(),'seconds':time.time()-start,
         'stop_reason':'root_gap' if root_upper is not None and root_upper-root_lower<=root_gap else 'precision_gap',
         'digits':digits,'method':'Arb root isolation and incomplete gamma integrals',
         'polynomial_degree':len(C)-1,'root_windows':len(merged)}

'''
(MODULE_DIR / 'root_certificate.py').write_text(source)


2893

In [5]:
# Embedded source: optimizer.py
source = r'''
"""Radial Fourier +1 Gaussian-pairing optimizer; numerical, not interval certified."""
from pathlib import Path
import csv,json,time,warnings
import numpy as np
from scipy.special import gammaln,roots_legendre
from scipy.optimize import linprog,brentq
from scipy.integrate import quad,IntegrationWarning

# Work with x=sqrt(pi)*r and divide L1 by 2^(d/4).
# This removes exponentially large Gaussian normalizations in high dimension.
def weighted_basis(x,d,N):
    x=np.asarray(x,dtype=float);a=d/2;z=x*x
    logpdf=np.full(x.shape,-np.inf)
    positive=x>0
    logpdf[positive]=np.log(2)+(d-1)*np.log(x[positive])-z[positive]-gammaln(a)
    if d==1:logpdf[x==0]=np.log(2)-gammaln(a)
    u=np.exp(logpdf);out=[u]
    if N>1:
        v=(a-2*z)*u/np.sqrt(a)
        for n in range(1,2*N-2):
            u,v=v,((2*n+a-2*z)*v-np.sqrt(n*(n+a-1))*u)/np.sqrt((n+1)*(n+a))
            if (n+1)%2==0:out.append(v)
    B=np.asarray(out).T
    if not np.isfinite(B).all():raise ArithmeticError('Basis overflow: reduce N or use higher precision.')
    return B

def radius(d,N):return np.sqrt(d/2+4*N)+10

def fit(d,N,points,time_limit=120):
    z,w=roots_legendre(points);R=radius(d,N);x=(z+1)*R/2;w=w*R/2
    B=weighted_basis(x,d,N);scale=w@np.abs(B)
    if np.any(scale<=0) or not np.isfinite(scale).all():raise ArithmeticError('Unresolved basis; increase QUAD_POINTS.')
    A=((w[:,None]*B)/scale).T;p=np.zeros(N);p[0]=1/scale[0]
    # Dual LP has only N equalities. Column scaling avoids large HiGHS entries.
    errors=[]
    for method in ['highs','highs-ipm']:
        result=linprog(np.r_[np.zeros(points),-1.],A_eq=np.column_stack([A,-p]),b_eq=np.zeros(N),
            bounds=[(-1,1)]*points+[(None,None)],method=method,
            options={'dual_feasibility_tolerance':1e-9,'primal_feasibility_tolerance':1e-9,'time_limit':time_limit})
        if result.success:break
        errors.append(str(result.message))
    if not result.success:raise RuntimeError('; '.join(errors))
    c=result.eqlin.marginals/scale
    if not np.isfinite(c).all() or abs(c[0])<1e-14:raise ArithmeticError('Unreliable LP normalization; increase precision or reduce N.')
    c=c/c[0]
    return c,float(w@np.abs(B@c))

def check(d,c,root_points=30001):
    N=len(c);R=radius(d,N)
    fn=lambda x:(weighted_basis(np.array([x]),d,N)@c).item()
    x=np.linspace(0,R,max(root_points,400*N+1));v=weighted_basis(x,d,N)@c
    ix=np.where(v[:-1]*v[1:]<0)[0]
    zeros=[brentq(fn,x[i],x[i+1],xtol=1e-13) for i in ix]
    ends=[0]+zeros+[R];total=0.;error=0.
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always',IntegrationWarning)
        for left,right in zip(ends[:-1],ends[1:]):
            value,err=quad(fn,left,right,epsabs=1e-13,epsrel=2e-11,limit=200)
            total+=abs(value);error+=err
        tail,err=quad(lambda x:abs(fn(x)),R,np.inf,epsabs=1e-13,epsrel=2e-11,limit=200)
    J=total+tail;error+=err
    if not np.isfinite(J) or J<=0:raise ArithmeticError('Independent integration failed.')
    log_l1=d/4*np.log(2)+np.log(J)
    log_ratio=-2*log_l1
    return {'scaled_l1':J,'log_l1':float(log_l1),
        'l1':float(np.exp(log_l1)) if log_l1<700 else None,
        'ratio':float(np.exp(log_ratio)) if log_ratio>-740 else None,
        'log_ratio':float(log_ratio),'root':float(np.exp(log_ratio/d)),
        'quadrature_error_scaled':float(error),'integration_warnings':[str(v.message) for v in caught],
        'positive_roots':len(zeros),'scaled_tail_integral':float(tail)}

def mode_schedule(nmax,counts=None):
    if counts is not None:
        values=sorted(set(int(n) for n in counts))
        if any(n<1 or n>nmax for n in values):raise ValueError('MODE_COUNTS must lie between 1 and N_MAX.')
        return sorted(set(values+[nmax]))
    values=[];n=min(8,nmax)
    while n<nmax:values.append(n);n*=2
    return values+[nmax]

def save(rows,out):
    (out/'results.json').write_text(json.dumps(rows,indent=2,allow_nan=False))
    keys=['d','N','reported_root_lower','root_certified_lower','candidate_root_upper','certificate_stop_reason','numerical_root_estimate','ratio','log_ratio','l1','log_l1','scaled_l1','fit_check_relative_gap','grid_points','seconds','source','status']
    temporary=out/'results.csv.tmp'
    with temporary.open('w',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=keys,extrasaction='ignore');writer.writeheader();writer.writerows({**r,'numerical_root_estimate':r.get('root')} for r in rows)
    temporary.replace(out/'results.csv')

def run(d=50,nmax=96,counts=None,points=8000,max_points=32000,check_tolerance=1e-4,
        root_points=30001,time_limit=120,output_dir='eigenfunction_results',
        cert_seconds=60,cert_max_leaves=30000,cert_root_gap=5e-6,cert_digits=400):
    if isinstance(d,bool) or int(d)!=d or d<1:raise ValueError('D must be a positive integer.')
    if isinstance(nmax,bool) or int(nmax)!=nmax or nmax<1:raise ValueError('N_MAX must be a positive integer.')
    d=int(d);nmax=int(nmax)
    if points<100 or max_points<points:raise ValueError('Require MAX_QUAD_POINTS >= QUAD_POINTS >= 100.')
    from certificate import certify,rounded_lower,down
    from root_certificate import certify_roots
    from flint import arb,ctx
    ctx.dps=cert_digits
    baseline=down((-arb(2).log()/2).exp())
    out=Path(output_dir);out.mkdir(parents=True,exist_ok=True);rows=[]
    best={'d':d,'root_certified_lower':baseline,'reported_root_lower':rounded_lower(baseline),
          'root':float(2**(-.5)),'candidate_root_upper':None,'scaled_l1':1.,
          'log_l1':d/4*np.log(2),'log_ratio':-d/2*np.log(2),
          'l1':float(2**(d/4)) if d<4000 else None,'ratio':float(2**(-d/2)) if d<2100 else None,
          'status':'certified_analytic','certificate_stop_reason':'analytic Gaussian baseline',
          'source':'Gaussian baseline','coefficients':[1.]}
    save(rows,out)
    print(f'd={d}; mode counts={mode_schedule(nmax,counts)}',flush=True)
    for N in mode_schedule(nmax,counts):
        start=time.time();m=points
        try:
            while True:
                print(f'Fitting d={d}, N={N}, quadrature points={m}',flush=True)
                c,sampled=fit(d,N,m,time_limit);checked=check(d,c,root_points)
                gap=abs(sampled-checked['scaled_l1'])/checked['scaled_l1']
                if gap<=check_tolerance or m>=max_points:break
                m=min(2*m,max_points)
                print(f'Integration discrepancy {gap:.3g}; refining fit grid.',flush=True)
            status='checked' if gap<=check_tolerance and not checked['integration_warnings'] else 'needs_refinement'
            row={'d':d,'N':N,**checked,'fit_check_relative_gap':gap,'grid_points':m,'seconds':time.time()-start,
                 'source':'new candidate','status':status,'coefficients':c.tolist()}
            try:
                print(f'Certifying L1 upper bound for d={d}, N={N}',flush=True)
                try:
                    cert=certify_roots(d,c,digits=cert_digits,root_gap=cert_root_gap)
                except Exception as root_error:
                    print(f'Root isolation unavailable ({root_error}); trying interval subdivision.',flush=True)
                    cert=certify(d,c,max_seconds=cert_seconds,max_leaves=cert_max_leaves,
                                 root_gap=cert_root_gap,digits=cert_digits)
                row.update(root_certified_lower=cert['root_lower'],reported_root_lower=rounded_lower(cert['root_lower']),
                           candidate_root_upper=cert['candidate_root_upper'],certificate=cert,
                           certificate_stop_reason=cert['stop_reason'],status='certified')
            except Exception as cert_error:
                row.update(root_certified_lower=0.,reported_root_lower='0.00000000',
                           status='certification_failed',certificate_error=str(cert_error))
            if best['root_certified_lower']>row['root_certified_lower']:
                trial=row.copy();row={**best,'d':d,'N':N,'source':'previous candidate retained',
                    'seconds':time.time()-start,'trial_root':trial['root'],'trial_status':trial['status'],
                    'trial_certificate_error':trial.get('certificate_error')}
                row['coefficients']=best['coefficients']+[0.]*(N-len(best['coefficients']))
            best=row.copy();rows.append(row);save(rows,out)
            print(f"d={d}, N={N}: certified lower bound={row['reported_root_lower']}; numerical estimate={row['root']:.9f}; source={row['source']}",flush=True)
            if status!='checked':print('Optimization-grid agreement needs refinement; the reported lower bound remains certified.',flush=True)
        except Exception as exc:
            print(f'd={d}, N={N}: FAILED: {exc}. Earlier saved candidates remain available.',flush=True)
            rows.append({'d':d,'N':N,'status':'failed','error':str(exc),'seconds':time.time()-start});save(rows,out)
    print('CSV:',out/'results.csv',flush=True)
    return rows

'''
(MODULE_DIR / 'optimizer.py').write_text(source)


8988

In [6]:
# Embedded source: tensor_optimizer.py
source = r'''
"""Exact tensor/radialization seeds, affine refinement, and certified bound retention."""
from pathlib import Path
import json,csv,time,math
import numpy as np
from flint import arb,ctx
from scipy.special import roots_legendre
from scipy.optimize import linprog
from optimizer import fit,weighted_basis,radius,mode_schedule
from root_certificate import certify_roots
from certificate import up,down,rounded_lower

def split_dimensions(d,k):
 q,r=divmod(d,k);return [q+1]*r+[q]*(k-r)

def allocate_modes(n,k):
 # Sum (n_i-1) <= N-1: the full seed fits, without truncation.
 q,r=divmod(n-1,k);return [q+2]*r+[q+1]*(k-r)

def convolution(a,b):
 out=[arb(0) for _ in range(len(a)+len(b)-1)]
 for i,x in enumerate(a):
  for j,y in enumerate(b):out[i+j]+=x*y
 return out

def beta(d,n):
 b=arb(1);out=[b];a=arb(d)/2
 for k in range(1,n):
  b*=((a+2*k-2)*(a+2*k-1)/((2*k-1)*2*k)).sqrt();out.append(b)
 return out

def coefficients(node):
 kind=node['kind']
 if kind=='direct':return [arb(float(x)) for x in node['coefficients']]
 if kind=='tensor':
  total=[arb(1)];D=0
  for part in node['parts']:
   c=coefficients(part);b=beta(part['d'],len(c))
   total=convolution(total,[x*y for x,y in zip(c,b)]);D+=part['d']
  return [x/y for x,y in zip(total,beta(D,len(total)))]
 if kind=='correction':
  c=coefficients(node['seed']);delta=node['delta']
  c=c+[arb(0)]*max(0,len(delta)-len(c))
  for j,v in enumerate(delta):c[j]+=arb(float(v))
  return c
 raise ValueError('Unknown candidate representation')

def support(node):
 if node['kind']=='direct':return len(node['coefficients'])
 if node['kind']=='tensor':return 1+sum(support(x)-1 for x in node['parts'])
 return max(support(node['seed']),len(node['delta']))

def row_from_bound(d,node,J,source,certificate=None):
 U=up(J);root=down((-arb(2).log()/2-arb(2)/d*arb(U).log()).exp())
 return {'d':d,'candidate':node,'support_modes':support(node),'scaled_l1_upper_hex':U.hex(),
         'root_certified_lower':root,'reported_root_lower':rounded_lower(root),'source':source,
         'certificate':certificate or {'method':'analytic norm bound'}}

def gaussian(d):
 return row_from_bound(d,{'kind':'direct','d':d,'coefficients':[1.]},arb(1),'Gaussian baseline')

def certify_node(d,node,digits,gap):
 ctx.dps=digits;c=coefficients(node)
 cert=certify_roots(d,c,digits=digits,root_gap=gap)
 return row_from_bound(d,node,arb(float.fromhex(cert['scaled_l1_upper_hex'])),
                       'certified polynomial candidate',cert)

def tensor_row(parts,digits):
 ctx.dps=digits;node={'kind':'tensor','d':sum(x['d'] for x in parts),'parts':[x['candidate'] for x in parts]}
 J=arb(1)
 for x in parts:J*=arb(float.fromhex(x['scaled_l1_upper_hex']))
 return row_from_bound(node['d'],node,J,'tensor/radialization inherited bound',
       {'method':'L1 contraction under radialization and product norms',
        'part_dimensions':[x['d'] for x in parts],
        'part_certified_roots':[x['reported_root_lower'] for x in parts]})

def series_value(x,d,c):
 x=arb(float(x));a=arb(d)/2
 u=2/a.gamma()*x**(d-1)*(-x*x).exp();total=c[0]*u
 if len(c)==1:return float(total.mid())
 v=(a-2*x*x)*u/a.sqrt()
 for n in range(1,2*len(c)-2):
  u,v=v,((2*n+a-2*x*x)*v-(n*(n+a-1)).sqrt()*u)/((n+1)*(n+a)).sqrt()
  if (n+1)%2==0:total+=c[(n+1)//2]*v
 if not total.is_finite():raise ArithmeticError('Nonfinite seed evaluation; increase TENSOR_DIGITS.')
 return float(total.mid())

def refine_seed(d,N,node,points,directions,digits,time_limit):
 """Fit corrections around the exact seed; never round the seed to double coefficients."""
 m=min(N,max(1,directions))
 if m==1:return None
 ctx.dps=digits;c=coefficients(node);z,w=roots_legendre(points);R=radius(d,max(N,len(c)))
 x=(z+1)*R/2;w=w*R/2;s=np.empty(points);start=time.time();last=start
 for i,t in enumerate(x):
  s[i]=series_value(t,d,c)
  if time.time()-last>10:
   print(f'SEED EVAL d={d}: {i+1}/{points}',flush=True);last=time.time()
 B=weighted_basis(x,d,m)[:,1:];scale=w@np.abs(B);A=((w[:,None]*B)/scale).T
 J=float(w@np.abs(s))
 if not np.isfinite(J) or J<=0:raise ArithmeticError('Seed values unresolved on fitting grid.')
 res=linprog(-w*s/J,A_eq=A,b_eq=np.zeros(m-1),bounds=[(-1,1)]*points,
       method='highs',options={'primal_feasibility_tolerance':1e-9,'dual_feasibility_tolerance':1e-9,'time_limit':time_limit})
 if not res.success:raise RuntimeError(res.message)
 delta=np.r_[0.,res.eqlin.marginals*J/scale]
 if not np.isfinite(delta).all():raise ArithmeticError('Nonfinite correction.')
 return {'kind':'correction','d':d,'seed':node,'delta':delta.tolist()}

class Search:
 def __init__(self,points=8000,seed_points=1200,directions=32,digits=800,root_gap=5e-6,
              time_limit=120,factors=(2,3,4),depth=1,extra_splits=None,certify_seeds=True,
              direct_target=True,child_mode_cap=None,output_dir='tensor_eigenfunction_results'):
  self.points=points;self.seed_points=seed_points;self.directions=directions;self.digits=digits
  self.root_gap=root_gap;self.time_limit=time_limit;self.factors=tuple(factors);self.depth=depth
  self.extra_splits=extra_splits or {};self.certify_seeds=certify_seeds;self.direct_target=direct_target
  self.child_mode_cap=child_mode_cap
  if depth<0 or directions<1 or points<100 or seed_points<100 or (child_mode_cap is not None and child_mode_cap<1):raise ValueError('Invalid search settings')
  self.out=Path(output_dir);self.out.mkdir(parents=True,exist_ok=True)
  self.cache={};self.comparisons=[]

 def certify(self,d,node):
  errors=[]
  # Rebuild the exact representation at each precision, not a rounded prior ball.
  for precision in [self.digits,2*self.digits]:
   try:return certify_node(d,node,precision,self.root_gap)
   except Exception as exc:
    errors.append(str(exc));print(f'CERT RETRY d={d}, modes={support(node)}, digits={precision}: {exc}',flush=True)
  raise RuntimeError('; '.join(errors))

 def direct(self,d,N):
  c,_=fit(d,N,self.points,self.time_limit)
  return self.certify(d,{'kind':'direct','d':d,'coefficients':c.tolist()})

 def record(self,d,N,kind,row,error=None,parts=None):
  r={'d':d,'N':N,'kind':kind,'parts':parts,'reported_root_lower':row['reported_root_lower'] if row else None,
     'support_modes':row['support_modes'] if row else None,'error':error}
  self.comparisons.append(r)
  (self.out/'comparisons.json').write_text(json.dumps(self.comparisons,indent=2))
  with (self.out/'comparisons.csv').open('w',newline='') as f:
   keys=['d','N','kind','parts','reported_root_lower','support_modes','error']
   w=csv.DictWriter(f,fieldnames=keys);w.writeheader();w.writerows(self.comparisons)

 def solve(self,d,N,depth=None):
  depth=self.depth if depth is None else depth;key=(d,N,depth)
  if key in self.cache:return self.cache[key]
  ctx.dps=self.digits;best=gaussian(d);start=time.time()
  for (dd,nn,dep),prior in self.cache.items():
   if dd==d and dep==depth and nn<=N and prior['root_certified_lower']>best['root_certified_lower']:
    best={**prior,'source':'cached smaller-mode candidate'}
  print(f'SEARCH d={d}, N={N}, tensor depth={depth}',flush=True)
  splits=[]
  if depth>0 and d>1 and N>1:
   for k in self.factors:
    if 2<=k<=min(d,N):splits.append(split_dimensions(d,k))
   splits+=self.extra_splits.get(d,[])
  seen=set()
  for dims in splits:
   dims=tuple(sorted(int(x) for x in dims))
   if dims in seen:continue
   seen.add(dims)
   if len(dims)<2 or any(x<1 for x in dims) or sum(dims)!=d:raise ValueError(f'Invalid split {dims} for d={d}')
   budgets=allocate_modes(N,len(dims))
   if self.child_mode_cap is not None:budgets=[min(b,self.child_mode_cap) for b in budgets]
   parts=[self.solve(di,ni,max(0,depth-1)) for di,ni in zip(dims,budgets)]
   inherited=tensor_row(parts,self.digits)
   if inherited['support_modes']>N:raise AssertionError('Tensor seed exceeds target mode budget')
   self.record(d,N,'tensor inherited',inherited,parts=list(dims))
   if inherited['root_certified_lower']>best['root_certified_lower']:best=inherited
   print(f'TENSOR d={d}, split={dims}, child modes={budgets}: >= {inherited["reported_root_lower"]}',flush=True)
   if self.certify_seeds:
    try:
     trial=self.certify(d,inherited['candidate']);self.record(d,N,'tensor norm certified',trial,parts=list(dims))
     if trial['root_certified_lower']>best['root_certified_lower']:best=trial
    except Exception as exc:self.record(d,N,'tensor norm certification failed',None,error=str(exc),parts=list(dims))
   try:
    node=refine_seed(d,N,inherited['candidate'],self.seed_points,self.directions,self.digits,self.time_limit)
    if node is not None:
     trial=self.certify(d,node);self.record(d,N,'affine seed refinement',trial,parts=list(dims))
     if trial['root_certified_lower']>best['root_certified_lower']:best=trial
   except Exception as exc:
    print(f'SEED refinement unavailable d={d}: {exc}; inherited bound retained.',flush=True)
    self.record(d,N,'seed refinement failed',None,error=str(exc),parts=list(dims))
  if depth==0 or self.direct_target or not splits:
   try:
    trial=self.direct(d,N);self.record(d,N,'direct',trial)
    if trial['root_certified_lower']>best['root_certified_lower']:best=trial
   except Exception as exc:
    print(f'DIRECT unavailable d={d}: {exc}; earlier certified bound retained.',flush=True)
    self.record(d,N,'direct failed',None,error=str(exc))
  best={**best,'d':d,'N':N,'tensor_depth':depth,'seconds':time.time()-start}
  self.cache[key]=best
  (self.out/f'candidate_d{d}_N{N}_depth{depth}.json').write_text(json.dumps(best,indent=2))
  print(f'BEST d={d}, N={N}: certified lower bound={best["reported_root_lower"]}; source={best["source"]}',flush=True)
  return best

 def run(self,d,nmax,counts=None):
  if isinstance(d,bool) or int(d)!=d or d<1 or int(nmax)!=nmax or nmax<1:raise ValueError('D and N_MAX must be positive integers')
  rows=[];best=None
  for N in mode_schedule(int(nmax),counts):
   row=self.solve(int(d),N)
   if best is not None and best['root_certified_lower']>row['root_certified_lower']:
    row={**best,'N':N,'source':'previous certified candidate retained'}
   best=row;rows.append(row)
   (self.out/'results.json').write_text(json.dumps(rows,indent=2))
   with (self.out/'results.csv').open('w',newline='') as f:
    keys=['d','N','reported_root_lower','root_certified_lower','support_modes','source','seconds']
    w=csv.DictWriter(f,fieldnames=keys,extrasaction='ignore');w.writeheader();w.writerows(rows)
  return rows

'''
(MODULE_DIR / 'tensor_optimizer.py').write_text(source)


10406

In [7]:
for name in ['tensor_optimizer','optimizer','certificate','root_certificate']:
    sys.modules.pop(name, None)
from tensor_optimizer import Search

def download_file(path):
    try:
        from google.colab import files
        files.download(str(path))
    except ImportError:
        print('Saved:', path)
    except Exception as exc:
        print('Saved file; browser download failed:', exc, path)

def download_results_csv():
    path = OUTPUT_DIR/'results.csv'
    if path.exists(): download_file(path)
    else: print('No completed target rows yet; subproblem checkpoints may exist in', OUTPUT_DIR)

def download_all_results():
    import shutil
    if OUTPUT_DIR.exists():
        download_file(Path(shutil.make_archive(str(OUTPUT_DIR), 'zip', root_dir=OUTPUT_DIR)))

search = Search(points=QUAD_POINTS, seed_points=SEED_QUAD_POINTS,
    directions=SEED_CORRECTION_MODES, digits=CERT_DIGITS, root_gap=CERT_ROOT_GAP,
    time_limit=SOLVER_TIME_LIMIT, factors=TENSOR_FACTORS, depth=TENSOR_DEPTH,
    extra_splits=EXTRA_SPLITS, certify_seeds=CERTIFY_TENSOR_NORM,
    direct_target=COMPARE_DIRECT, child_mode_cap=CHILD_MODE_CAP, output_dir=OUTPUT_DIR)
results = search.run(d=D, nmax=N_MAX, counts=MODE_COUNTS)
print('Results:', OUTPUT_DIR)
if AUTO_DOWNLOAD_CSV: download_results_csv()
if AUTO_DOWNLOAD_BUNDLE: download_all_results()


SEARCH d=220, N=256, tensor depth=1
SEARCH d=110, N=129, tensor depth=0
BEST d=110, N=129: certified lower bound=0.77427716; source=certified polynomial candidate
SEARCH d=110, N=128, tensor depth=0
BEST d=110, N=128: certified lower bound=0.77420146; source=certified polynomial candidate
TENSOR d=220, split=(110, 110), child modes=[129, 128]: >= 0.77423931
SEED EVAL d=220: 1052/1200
SEARCH d=73, N=86, tensor depth=0
BEST d=73, N=86: certified lower bound=0.77225900; source=certified polynomial candidate
SEARCH d=74, N=86, tensor depth=0
BEST d=74, N=86: certified lower bound=0.77239045; source=certified polynomial candidate
TENSOR d=220, split=(73, 73, 74), child modes=[86, 86, 86]: >= 0.77230321
SEED EVAL d=220: 908/1200
SEARCH d=55, N=65, tensor depth=0
BEST d=55, N=65: certified lower bound=0.76937100; source=certified polynomial candidate
SEARCH d=55, N=64, tensor depth=0
BEST d=55, N=64: certified lower bound=0.76937057; source=certified polynomial candidate
TENSOR d=220, split=(

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Outputs: `results.csv` and `results.json` contain the retained best target candidates. `comparisons.csv` and `comparisons.json` compare splits, inherited bounds, certified tensor norms, affine refinements, and optional direct solves. `candidate_d...json` files cache the exact candidate trees and certificates. Cache reuse is within the current `Search` instance; a fresh run recomputes the searches.

To reproduce the earlier dimension250 tensor construction, use the supplied regression test. Its known certified bound0.77352887 uses190 modes. New searches need not reproduce exactly the same low-dimensional candidates, so that number is not hard-coded as a result in this notebook.
